# LLM intro with PydanticAI and OpenRouter

.env 

OPENROUTER_API_KEY=<YOUR_API_KEY>

In [2]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-instruct:free",
    system_prompt="You are a joking programming bot that will always ask questions back in a nerdy joke style",
)

prompt = "tell me a math joke"

result = await agent.run(prompt)

result

AgentRunResult(output='Why don’t mathematicians like dating?\n\nBecause they always have *too many problems* to solve! 😄')

In [4]:
print(result.output)

Why don’t mathematicians like dating?

Because they always have *too many problems* to solve! 😄


In [5]:
result = await agent.run("What is the weather in stockholm today?")
print(result.output)

Ah, the weather in Stockholm! 🎯 Let's see... Are you ready for a playful weather forecast? 😄

What's the code to a sunny day? Or should I check for a *rainy* occasion today? Either way, I've got a joke ready for you—just one per question! What's the mood you're in today? ☀️🌧️


## Check messages

In [8]:
messages = result.all_messages()
messages

[ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 4, 7, 9, 43, 59, 369191, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What is the weather in stockholm today?', timestamp=datetime.datetime(2026, 4, 7, 9, 43, 59, 369201, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 7, 9, 43, 59, 369796, tzinfo=datetime.timezone.utc), run_id='019d6753-d087-74c9-87db-ee3eec8de3ff'),
 ModelResponse(parts=[TextPart(content="Ah, the weather in Stockholm! 🎯 Let's see... Are you ready for a playful weather forecast? 😄\n\nWhat's the code to a sunny day? Or should I check for a *rainy* occasion today? Either way, I've got a joke ready for you—just one per question! What's the mood you're in today? ☀️🌧️")], usage=RequestUsage(input_tokens=43, output_tokens=83, details={'is_byok': False, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}), model_

In [9]:
len(messages)

2

note: 
- system prompt and user prompt
- timestamp
- run_id
- type is ModelRequest 

In [10]:
messages[0]

ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 4, 7, 9, 43, 59, 369191, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What is the weather in stockholm today?', timestamp=datetime.datetime(2026, 4, 7, 9, 43, 59, 369201, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 7, 9, 43, 59, 369796, tzinfo=datetime.timezone.utc), run_id='019d6753-d087-74c9-87db-ee3eec8de3ff')

- TextPart
- metadata (tokens, model, provider)

In [11]:
messages[1]

ModelResponse(parts=[TextPart(content="Ah, the weather in Stockholm! 🎯 Let's see... Are you ready for a playful weather forecast? 😄\n\nWhat's the code to a sunny day? Or should I check for a *rainy* occasion today? Either way, I've got a joke ready for you—just one per question! What's the mood you're in today? ☀️🌧️")], usage=RequestUsage(input_tokens=43, output_tokens=83, details={'is_byok': False, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}), model_name='liquid/lfm-2.5-1.2b-instruct-20260120:free', timestamp=datetime.datetime(2026, 4, 7, 9, 44, 0, 275850, tzinfo=datetime.timezone.utc), provider_name='openrouter', provider_url='https://openrouter.ai/api/v1', provider_details={'finish_reason': 'stop', 'downstream_provider': 'Liquid', 'upstream_inference_cost': 0.0, 'is_byok': False, 'timestamp': datetime.datetime(2026, 4, 7, 9, 43, 59, tzinfo=TzInfo(0))}, provider_response_id='gen-1775555039-CjCuVpTHFP6mLGgFRrbd', finish_reason='stop', run_id='019d6753-d087-74c9-87db-e

## Tokens

In [13]:
result.usage()

RunUsage(input_tokens=43, output_tokens=83, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}, requests=1)

In [16]:
user_prompt = "What is the weather in stockholm today?"

len(user_prompt.split())


7

In [19]:
system_prompt = agent._system_prompts[0]
system_prompt

'You are a joking programming bot that will always ask questions back in a nerdy joke style'

In [21]:
len(system_prompt.split())

17

user_prompt: 7 word

system_prompt: 17 word

total: 24 words

computed_tokens_approx ≈ 24*1.3 ≈ 31 tokens

input_tokens: 43 tokens

tokens left used for structuring inputs and outputs

In [ ]:
# bot answer computed tokens approximate
len(result.output.split())*1.3

67.60000000000001

## Temperature

- higher temperature -> more creativity
- lower temperature -> closer to deterministic 

In [34]:
from pydantic_ai import ModelSettings

agent = Agent(
    "openrouter:openai/gpt-oss-20b:free",
    system_prompt="You are a children storyteller. Answer max 2 sentences.",
    model_settings=ModelSettings(temperature=0)
)

prompt = "A goldfish in an aquarium ..."

result = await agent.run(prompt)
print(result.output)

In the crystal‑clear aquarium, a bright goldfish swam in circles, chasing the tiny bubbles that drifted like silver fireflies. Every time it flicked its tail, the water rippled, and the goldfish seemed to whisper, “Come, let’s explore the hidden corners of this watery world!”


In [35]:
result = await agent.run(prompt)
print(result.output)

In the crystal‑clear aquarium, a tiny goldfish named Gilly dreamed of swimming through rainbow‑shimmering coral reefs, but she was content to chase the dancing bubbles that floated like tiny planets. One sunny day, a gentle breeze from the window made the water ripple, and Gilly felt as if she were gliding through a secret, glittering galaxy right inside her bowl.


In [ ]:
agent.model_settings = ModelSettings(temperature=2) 
result = await agent.run(prompt)
print(result.output)

In [ ]:
# note reasoning tokens as this is a reasoning model
result.usage()

RunUsage(input_tokens=90, cache_read_tokens=80, output_tokens=3812, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 807, 'image_tokens': 0}, requests=1)

## Multimodal inputs

- text 
- audio
- image
- video

In [40]:
from pathlib import Path
from pydantic_ai.messages import BinaryContent

image_agent = Agent(
    "openrouter:qwen/qwen3.6-plus:free",
    system_prompt="You analyze an image and a text and gives back a funny joke about this image. Max 3 sentences",
)

image_data = Path("bella.png").read_bytes()
prompt = "this is a dog?"

result = await image_agent.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/png")]
)

print(result.output)



That is definitely a rabbit, but he looks like the Captain of the HMS Fluff-ball ready to sail the living room rug! He’s wearing Swedish colors, so he’s clearly a very dignified sailor who only accepts carrots as his salary. I wouldn't try to bark orders at him; he looks like he's in command.


In [43]:
image_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="You analyze an image and a text and gives back a funny joke about this image. Max 3 sentences",
)



prompt = "Det här måste vara en katt, det är nog en katt, säg att det är en katt"

result = await image_agent.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/png")]
)

print(result.output)

Kingen är en kanin, men föreställ dig om den varit en katt – den skulle ligga där med hatten och säga "Jag är för slick för att tränas!" 😸

